# MoViT-SAM: patient-level, leakage-free evaluation of Mpox differential diagnosis (MSLD v2.0)

This notebook reproduces every analysis reported in the manuscript:

| Manuscript item | Section below |
|---|---|
| Data integrity audit (pHash, patient IDs, ITA) | 4 |
| Patient-level split, Table 1 | 5 |
| Focal loss + label smoothing + Mixup, SAM / H-SAM | 6–9 |
| Five-fold stratified-group CV, pooled OOF, Tables 3–5, Figs 3 & 5 | 11–13 |
| Calibration (cross-fitted temperature scaling), Fig 4 | 14 |
| Clinical evaluation: screening, PPV/NPV, DCA, error taxonomy, Table 7, Fig 6 | 15 |
| Leakage quantification (Protocols A and B), Table 2 | 16 |
| Single-run ablation, Table 6 | 17 |
| TFLite FP32 / INT8 deployment, Table 8 | 18 |

**Leakage safeguards (all enforced in code):** patient-disjoint splits; augmentation only
on training batches; early stopping and learning-rate scheduling use an *inner* validation
split carved from the training patients, never the evaluation fold; temperature scaling is
cross-fitted so no fold is calibrated on its own predictions.

Run order: Section 0 → *Restart & Run All*. Results are written to `./results/`; each CV
fold is checkpointed so an interrupted Kaggle session can resume.

In [ ]:
# ============================================================
# 0 - Install dependencies (Kaggle: Internet = ON)
# ============================================================
import os, subprocess, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # Keras-2 API required by TFViTModel

def pip(args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("pip FAILED for", args, "\n", (r.stderr or r.stdout)[-1500:])

if not os.environ.get("MOVIT_SKIP_INSTALL"):
    # tf-keras must match the installed TensorFlow (an unpinned install pulls a newer TensorFlow).
    from importlib.metadata import version as _v
    _tf = None
    for _name in ("tensorflow", "tensorflow-cpu"):
        try: _tf = _v(_name); break
        except Exception: pass
    if _tf:
        _mm = ".".join(_tf.split(".")[:2])
        pip([f"tf-keras=={_mm}.*"])
    else:
        pip(["tf-keras"])
    # transformers 4.46.3 still provides TFViTModel and has pre-built wheels (tokenizers 0.20.3) for
    # current Python versions, including 3.13, which Kaggle now uses.
    pip(["transformers==4.46.3", "tokenizers==0.20.3", "huggingface_hub==0.24.6"])
    for p in ["scikit-learn>=1.3.0", "matplotlib>=3.7.0", "seaborn>=0.12.0",
              "opencv-python>=4.8.0", "pandas>=2.0.0", "scipy>=1.11.0",
              "ImageHash>=4.3.1", "kaggle", "Pillow"]:
        pip([p])
    r = subprocess.run([sys.executable, "-c", "from transformers import TFViTModel; import transformers; print(transformers.__version__)"],
                       capture_output=True, text=True, env={**os.environ, "TF_USE_LEGACY_KERAS": "1", "TF_CPP_MIN_LOG_LEVEL": "3"})
    if r.returncode != 0:
        raise RuntimeError("TFViTModel import check failed:\n" + r.stderr[-2000:])
    print("Dependencies installed. transformers", r.stdout.strip().splitlines()[-1], "| TFViTModel import OK")

In [ ]:
# ============================================================
# 1 - Imports, reproducibility, configuration
# ============================================================
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
import re, random, json, time, platform, itertools
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import imagehash
from scipy import stats as sp_stats
from scipy.optimize import minimize_scalar

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import (StratifiedGroupKFold, GroupShuffleSplit,
                                     StratifiedShuffleSplit)
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             roc_auc_score, precision_recall_curve,
                             average_precision_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

sns.set_style("whitegrid")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# TINY_TEST: pipeline check on synthetic images with tiny random backbones (no downloads).
# SMOKE_TEST: real data and backbones, very short training.  Both False = full experiment.
TINY_TEST  = bool(int(os.environ.get("MOVIT_TINY_TEST", "0")))
SMOKE_TEST = False

# Label order. Monkeypox MUST stay at index 0: the deployed TFLite model and the
# external-validation notebook read P(Monkeypox) from output index 0.
CLASS_NAMES = ["Monkeypox", "Chickenpox", "Measles", "Cowpox", "HFMD", "Healthy"]
NUM_CLASSES = len(CLASS_NAMES)
MPOX_IDX    = CLASS_NAMES.index("Monkeypox")
IMG_SIZE, CHANNELS = 224, 3

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
INNER_VAL_FRAC = 0.15            # inner early-stopping split inside each training fold

quick = TINY_TEST or SMOKE_TEST
BATCH_SIZE = 16 if quick else 32
EPOCHS     = 1  if quick else 40
PATIENCE   = 1  if quick else 8
K_FOLDS    = 2  if quick else 5
LR_HEAD, LR_MOBILENET, LR_VIT, LR_EFFNET = 5e-4, 1e-3, 1e-4, 1e-3

FOCAL_GAMMA     = 2.0
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA     = 0.2
RHO, ALPHA      = 0.05, 0.5      # SAM radius; H-SAM adaptive coefficient

N_BOOT          = 200 if quick else 2000
PREVALENCES     = [0.01, 0.05, 0.15]          # + dataset prevalence (added below)
SENS_TARGETS    = [0.90, 0.95]
DCA_THRESHOLDS  = np.arange(0.01, 0.601, 0.01)
PHASH_THRESHOLD = 8
AUG_COPIES_B    = 6                           # Protocol B: copies per original

DATA_ROOT   = Path(os.environ.get("MOVIT_DATA_ROOT", "./data/msld_v2"))
RESULTS_DIR = Path(os.environ.get("MOVIT_RESULTS", "./results")); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = Path(os.environ.get("MOVIT_MODELS", "./saved_models")); MODELS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = RESULTS_DIR / "cv_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

HARDWARE = {
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "platform": platform.platform(), "processor": platform.processor() or platform.machine(),
    "cpu_count": os.cpu_count(), "gpu": [g.name for g in gpus] or "none",
}
from importlib.metadata import version as _ver
for _pkg in ("transformers", "tf-keras", "scikit-learn", "numpy"):
    try: HARDWARE[_pkg] = _ver(_pkg)
    except Exception: pass
try:
    HARDWARE["cpu_model"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo")
                                 if l.startswith("model name"))
except Exception:
    pass
(RESULTS_DIR / "hardware.json").write_text(json.dumps(HARDWARE, indent=2))
if not gpus and not (TINY_TEST or SMOKE_TEST):
    raise RuntimeError("No GPU detected. On Kaggle: Settings > Accelerator > GPU T4 x2, then Save & Run All again. "
                       "Without a GPU the full run would take days.")
print(json.dumps(HARDWARE, indent=2))
print("TINY_TEST:", TINY_TEST, "| SMOKE_TEST:", SMOKE_TEST, "| classes:", CLASS_NAMES)

In [ ]:
# ============================================================
# 2 - Obtain MSLD v2.0
# ============================================================
# Dataset: https://www.kaggle.com/datasets/joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20
# On Kaggle, add it via "Add Input" and point MOVIT_DATA_ROOT (or DATA_ROOT) at it.
KAGGLE_SLUG = "joydippaul/mpox-skin-lesion-dataset-version-20-msld-v20"

def _has_images(root):
    return root.exists() and any(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in root.rglob("*"))

if TINY_TEST and not _has_images(DATA_ROOT):
    # Synthetic stand-in with MSLD-style filenames (class prefix + patient + image no.).
    rng = np.random.default_rng(SEED)
    prefix = {"Monkeypox": "MKP", "Chickenpox": "CHP", "Measles": "MSL",
              "Cowpox": "CWP", "HFMD": "HFMD", "Healthy": "HEALTHY"}
    for ci, cls in enumerate(CLASS_NAMES):
        d = DATA_ROOT / "Original Images" / cls; d.mkdir(parents=True, exist_ok=True)
        for pt in range(1, 9):
            base = rng.integers(40, 200, 3)
            for k in range(1, 3):
                img = np.clip(base + rng.normal(0, 25, (96, 96, 3)) + ci * 8, 0, 255).astype("uint8")
                Image.fromarray(img).save(d / f"{prefix[cls]}_{pt:02d}_{k:02d}.jpg")
elif not _has_images(DATA_ROOT):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    rc = os.system(f'kaggle datasets download -d {KAGGLE_SLUG} --unzip -p "{DATA_ROOT}"')
    if rc != 0:
        raise RuntimeError("Download failed: attach the dataset as a Kaggle input and set DATA_ROOT.")

In [ ]:
# ============================================================
# 3 - Locate ORIGINAL images, build catalogue, extract patient IDs
# ============================================================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
CLASS_ALIASES = {
    "monkeypox": "Monkeypox", "mpox": "Monkeypox", "mkp": "Monkeypox", "monkey pox": "Monkeypox",
    "chickenpox": "Chickenpox", "chicken pox": "Chickenpox", "chp": "Chickenpox",
    "measles": "Measles", "msl": "Measles",
    "cowpox": "Cowpox", "cow pox": "Cowpox", "cwp": "Cowpox",
    "hfmd": "HFMD", "hand foot mouth disease": "HFMD", "handfootmouth": "HFMD",
    "healthy": "Healthy", "normal": "Healthy", "hth": "Healthy",
}

def canonical(name):
    key = re.sub(r"\s+", " ", name.strip().lower().replace("-", " ").replace("_", " "))
    return CLASS_ALIASES.get(key) or CLASS_ALIASES.get(key.replace(" ", ""))

def extract_patient_id(cls, fname):
    """MSLD filenames encode the patient: e.g. MKP_01_01.jpg -> Monkeypox_0001.
    Images without a parsable patient number are treated as their own patient."""
    m = re.match(r"^([A-Za-z]+)[ _-]?(\d+)", Path(fname).stem)
    return f"{cls}_{int(m.group(2)):04d}" if m else f"{cls}_{Path(fname).stem}"

# MSLD v2.0 distributes its ORIGINAL images across several fold sub-folders (e.g. Train /
# Valid / Test inside each fold), so no single folder holds all of them. We therefore walk the
# whole dataset, skip anything under an "augmented" path, take the class from the nearest
# class-named ancestor folder, and de-duplicate by (class, filename) to recover the union of
# unique originals.
EXPECTED_ORIGINALS = 755   # MSLD v2.0 original images

def class_of(path):
    for parent in path.parents:
        c = canonical(parent.name)
        if c: return c
        if parent == DATA_ROOT: break
    return None

records = []
for f in sorted(DATA_ROOT.rglob("*")):
    if f.suffix.lower() not in IMG_EXT or "augment" in str(f).lower():
        continue
    cls = class_of(f)
    if cls is None: continue
    records.append({"path": str(f), "fname": f.name.lower(), "cls": cls,
                    "label": CLASS_NAMES.index(cls), "patient": extract_patient_id(cls, f.name)})
df = (pd.DataFrame(records).drop_duplicates(subset=["cls", "fname"])
        .sort_values(["label", "fname"]).reset_index(drop=True))
paths, labels, groups = df["path"].values, df["label"].values, df["patient"].values
print(df.groupby("cls").agg(images=("path", "size"), patients=("patient", "nunique"))
        .reindex(CLASS_NAMES).to_string())
print(f"TOTAL: {len(df)} images, {df['patient'].nunique()} patients")
if not TINY_TEST and len(df) != EXPECTED_ORIGINALS:
    raise RuntimeError(f"Found {len(df)} unique original images, expected {EXPECTED_ORIGINALS}. "
                       "Check DATA_ROOT points at the full MSLD v2.0 dataset before running.")

# Invalidate saved CV folds if the image set has changed since they were written.
import hashlib, shutil
FINGERPRINT = hashlib.sha256("\n".join(sorted(df["cls"] + "/" + df["fname"])).encode()).hexdigest()
_fp = CKPT_DIR / "dataset_fingerprint.txt"
if _fp.exists() and _fp.read_text().strip() != FINGERPRINT:
    print("Dataset changed since the saved CV folds were written: deleting old checkpoints.")
    shutil.rmtree(CKPT_DIR); CKPT_DIR.mkdir()
_fp.write_text(FINGERPRINT)

In [ ]:
# ============================================================
# 4 - Data integrity audit: perceptual hashing and skin tone (ITA)
# ============================================================
df["phash"] = [imagehash.phash(Image.open(p).convert("RGB")) for p in df["path"]]

def near_duplicate_pairs(hashes, thr=PHASH_THRESHOLD):
    h = list(hashes)
    return [(i, j, h[i] - h[j]) for i in range(len(h)) for j in range(i + 1, len(h))
            if h[i] - h[j] <= thr]

dup_pairs = near_duplicate_pairs(df["phash"])
cross_patient = [(i, j, d) for i, j, d in dup_pairs if groups[i] != groups[j]]
print(f"Near-duplicate pairs (Hamming <= {PHASH_THRESHOLD}): {len(dup_pairs)} "
      f"({len(cross_patient)} across different inferred patients)")
pd.DataFrame([{"a": paths[i], "b": paths[j], "hamming": d, "same_patient": groups[i] == groups[j]}
              for i, j, d in dup_pairs]).to_csv(RESULTS_DIR / "audit_near_duplicates.csv", index=False)

def ita_degrees(path):
    """Individual Typology Angle, ITA = arctan((L* - 50) / b*) in degrees, taken as the median
    over skin-coloured pixels (YCrCb skin mask, Cr 133-173, Cb 77-127). A proxy for skin tone,
    not a validated Fitzpatrick assessment."""
    rgb = np.array(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    mask = (ycc[..., 1] >= 133) & (ycc[..., 1] <= 173) & (ycc[..., 2] >= 77) & (ycc[..., 2] <= 127)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0] * 100.0 / 255.0
    b = lab[..., 2] - 128.0
    sel = mask & (b > 1.0)
    if sel.sum() < 50:
        return np.nan
    return float(np.median(np.degrees(np.arctan((L[sel] - 50.0) / b[sel]))))

df["ita"] = [ita_degrees(p) for p in df["path"]]
ita_tab = df.groupby("cls")["ita"].agg(["median", "count"]).reindex(CLASS_NAMES).round(1)
print("Median ITA by class:\n", ita_tab.to_string())
ita_tab.to_csv(RESULTS_DIR / "audit_ita_by_class.csv")

In [ ]:
# ============================================================
# 5 - Patient-level 70/15/15 split BEFORE augmentation  (Table 1)
# ============================================================
gss1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_RATIO, random_state=SEED)
train_idx, temp_idx = next(gss1.split(paths, labels, groups))
gss2 = GroupShuffleSplit(n_splits=1, train_size=VAL_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED)
v_rel, t_rel = next(gss2.split(paths[temp_idx], labels[temp_idx], groups[temp_idx]))
val_idx, test_idx = temp_idx[v_rel], temp_idx[t_rel]

G = [set(groups[i]) for i in (train_idx, val_idx, test_idx)]
assert G[0].isdisjoint(G[1]) and G[0].isdisjoint(G[2]) and G[1].isdisjoint(G[2]), "Patient leak!"

def split_near_dups(a_idx, b_idx):
    ha, hb = df["phash"].values[a_idx], df["phash"].values[b_idx]
    return sum(1 for x in hb if any(x - y <= PHASH_THRESHOLD for y in ha))

print("Patient-disjoint split verified.")
print("Test images with a near-duplicate in train:", split_near_dups(train_idx, test_idx))

def counts(idx):
    c = Counter(labels[idx]); return [c.get(k, 0) for k in range(NUM_CLASSES)]
def pcounts(idx):
    s = defaultdict(set)
    for i in idx: s[labels[i]].add(groups[i])
    return [len(s[k]) for k in range(NUM_CLASSES)]

all_idx = np.arange(len(df))
table1 = pd.DataFrame({"Class": CLASS_NAMES, "Total images": counts(all_idx),
                       "Patients": pcounts(all_idx), "Train images": counts(train_idx),
                       "Val images": counts(val_idx), "Test images": counts(test_idx)})
table1.loc[len(table1)] = ["TOTAL", len(df), df["patient"].nunique(), len(train_idx),
                           len(val_idx), len(test_idx)]
print(table1.to_string(index=False))
print(f"Held-out test set: {len(test_idx)} images from {len(G[2])} patients")
table1.to_csv(RESULTS_DIR / "table1_split.csv", index=False)

In [ ]:
# ============================================================
# 6 - Input pipeline: raw [0,255] RGB, on-the-fly augmentation, class weights, Mixup
# ============================================================
# Every model receives raw [0,255] pixels and applies its backbone's own preprocessing
# internally, so the exported TFLite model also expects raw [0,255] RGB input.
AUTOTUNE = tf.data.AUTOTUNE
_AUG = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, fill_mode="reflect", seed=SEED),  # +/-0.15 of a full turn (+/-54 deg)
    layers.RandomZoom(0.15, fill_mode="reflect", seed=SEED),      # +/-15 %
    layers.RandomContrast(0.10, seed=SEED),                       # +/-10 %
], name="augmentation")

def class_weights_for(idx):
    present = np.unique(labels[idx])
    w = compute_class_weight("balanced", classes=present, y=labels[idx])
    cw = {int(c): float(x) for c, x in zip(present, w)}
    return {k: cw.get(k, 1.0) for k in range(NUM_CLASSES)}

def _decode(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=CHANNELS, expand_animations=False)
    img = tf.ensure_shape(img, [None, None, CHANNELS])
    return tf.cast(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]), tf.float32)

def _sample_beta(alpha):
    a = tf.random.gamma([], alpha); b = tf.random.gamma([], alpha)
    return a / (a + b)

def _mixup(x, y, w):
    lam = _sample_beta(MIXUP_ALPHA)
    perm = tf.random.shuffle(tf.range(tf.shape(x)[0]))
    return (lam * x + (1 - lam) * tf.gather(x, perm),
            lam * y + (1 - lam) * tf.gather(y, perm),
            lam * w + (1 - lam) * tf.gather(w, perm))

def make_ds(idx, training=False, cw=None, batch=BATCH_SIZE, path_arr=None, label_arr=None):
    """Training: shuffle -> batch -> augment -> Mixup, class weights as sample weights.
    Evaluation: original images only, in the given order."""
    P = (path_arr if path_arr is not None else paths)[idx]
    Y = (label_arr if label_arr is not None else labels)[idx]
    W = np.array([cw[int(l)] for l in Y], np.float32) if cw else np.ones(len(Y), np.float32)
    ds = tf.data.Dataset.from_tensor_slices((P, Y, W))
    if training:
        ds = ds.shuffle(len(P), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l, w: (_decode(p), tf.one_hot(l, NUM_CLASSES), w),
                num_parallel_calls=AUTOTUNE).batch(batch)
    if training:
        ds = ds.map(lambda x, y, w: _mixup(_AUG(x, training=True), y, w), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

In [ ]:
# ============================================================
# 7 - Loss: focal loss (gamma = 2) with label smoothing (eps = 0.05)
# ============================================================
# The class-balancing term alpha_t of the focal loss is supplied through the per-sample
# class weights built in Section 6.
def focal_loss(gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING):
    def loss(y_true, y_pred):
        y_true = y_true * (1.0 - smoothing) + smoothing / NUM_CLASSES
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        return tf.reduce_sum(tf.pow(1.0 - y_pred, gamma) * (-y_true * tf.math.log(y_pred)), axis=-1)
    loss.__name__ = "focal_loss"
    return loss

In [ ]:
# ============================================================
# 8 - Sharpness-aware training: plain (Adam) / SAM / H-SAM
# ============================================================
#   SAM  : eps = rho * g / ||g||_2                          (Foret et al., 2021)
#   H-SAM: eps = rho * g / ||g||_2 * (1 + alpha * r),  r = ||theta|| / (||theta|| + ||g||)
class SharpnessAwareModel(keras.Model):
    def __init__(self, inner, mode="sam", rho=RHO, alpha=ALPHA, **kw):
        super().__init__(**kw)
        self.inner, self.mode, self.rho, self.alpha = inner, mode, rho, alpha

    def call(self, x, training=False):
        return self.inner(x, training=training)

    def _perturb(self, grads, vars_):
        gnorm = tf.linalg.global_norm([g for g in grads if g is not None]) + 1e-12
        eps = []
        for g, v in zip(grads, vars_):
            if g is None: eps.append(None); continue
            e = self.rho * tf.cast(g, tf.float32) / gnorm
            if self.mode == "hsam":
                pn = tf.norm(tf.cast(v, tf.float32)) + 1e-12
                gl = tf.norm(tf.cast(g, tf.float32)) + 1e-12
                e = e * (1.0 + self.alpha * pn / (pn + gl))
            eps.append(e)
        return eps

    def train_step(self, data):
        x, y, sw = data
        tv = self.trainable_variables
        with tf.GradientTape() as tape:
            yp = self(x, training=True)
            loss = self.compiled_loss(y, yp, sample_weight=sw, regularization_losses=self.losses)
        g = tape.gradient(loss, tv)
        if self.mode != "none":
            eps = self._perturb(g, tv)
            for v, e in zip(tv, eps):
                if e is not None: v.assign_add(e)
            with tf.GradientTape() as tape2:
                yp2 = self(x, training=True)
                loss2 = self.compiled_loss(y, yp2, sample_weight=sw, regularization_losses=self.losses)
            g = tape2.gradient(loss2, tv)
            for v, e in zip(tv, eps):
                if e is not None: v.assign_sub(e)
        self.optimizer.apply_gradients(zip(g, tv))
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data[0], data[1]
        sw = data[2] if len(data) > 2 else None
        yp = self(x, training=False)
        self.compiled_loss(y, yp, sample_weight=sw)
        self.compiled_metrics.update_state(y, yp, sample_weight=sw)
        return {m.name: m.result() for m in self.metrics}

In [ ]:
# ============================================================
# 9 - Backbones (each applies its own preprocessing to raw [0,255] input)
# ============================================================
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406])
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225])

def _tiny_features(dim, name):
    """TINY_TEST stand-in: small random CNN with the same output width (no downloads)."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = layers.Rescaling(1 / 255.)(inp)
    x = layers.Conv2D(8, 5, strides=4, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dim, activation="relu")(x)
    return keras.Model(inp, x, name=name)

def mobilenet_features(name="mob"):
    if TINY_TEST: return _tiny_features(1280, name)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.mobilenet_v2.preprocess_input(inp)
    bb = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = layers.GlobalAveragePooling2D()(bb(x, training=False))
    return keras.Model(inp, x, name=name)

def effnet_features(name="eff"):
    if TINY_TEST: return _tiny_features(1280, name)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = keras.applications.efficientnet_v2.preprocess_input(inp)
    bb = keras.applications.EfficientNetV2B0(include_top=False, weights="imagenet",
                                             input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    bb.trainable = False
    x = layers.GlobalAveragePooling2D()(bb(x, training=False))
    return keras.Model(inp, x, name=name)

def vit_features(name="vit"):
    """Frozen ViT-B/16 -> 768-d CLS token. The backbone is not fine-tuned: only the classification
    head on top is trained (MoViT-SAM: 1,052,166 trainable of 89,699,398 parameters)."""
    if TINY_TEST: return _tiny_features(768, name)
    from transformers import TFViTModel
    vit = TFViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
    vit.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = (layers.Rescaling(1.0 / 255)(inp) - IMAGENET_MEAN) / IMAGENET_STD
    x = layers.Permute((3, 1, 2))(x)
    return keras.Model(inp, vit(x).last_hidden_state[:, 0, :], name=name)

In [ ]:
# ============================================================
# 10 - Model factory and training helpers
# ============================================================
def _compile(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=focal_loss(), metrics=["accuracy"])
    return model

def build_single(kind):
    feat, lr = {"mobilenet": (mobilenet_features, LR_MOBILENET),
                "effnet": (effnet_features, LR_EFFNET),
                "vit": (vit_features, LR_VIT)}[kind]
    f = feat()
    x = layers.Dropout(0.3)(layers.Dense(256, activation="relu")(f.output))
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    inner = keras.Model(f.input, out, name=f"{kind}_single")
    m = SharpnessAwareModel(inner, mode="none", name=f"{kind}_adam")
    _ = m(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))
    return _compile(m, lr)

def build_hybrid(mode="sam"):
    """MoViT: frozen MobileNetV2 (1280) || frozen ViT-B/16 (768) -> concat 2048
    -> Dense(512, ReLU) -> Dropout(0.3) -> Dense(6, softmax). mode: none | sam | hsam."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    fused = layers.Concatenate(name="fusion")([mobilenet_features("mob_stream")(inp),
                                               vit_features("vit_stream")(inp)])
    h = layers.Dropout(0.3)(layers.Dense(512, activation="relu", name="fc1")(fused))
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="out")(h)
    m = SharpnessAwareModel(keras.Model(inp, out, name=f"hybrid_{mode}"), mode=mode,
                            name=f"hybrid_{mode}_wrapper")
    _ = m(tf.zeros((1, IMG_SIZE, IMG_SIZE, CHANNELS)))
    return _compile(m, LR_HEAD)

MODELS = {   # name -> builder.  MoViT-SAM (standard SAM) is the proposed model.
    "MoViT-SAM":      lambda: build_hybrid("sam"),
    "Hybrid+Adam":    lambda: build_hybrid("none"),
    "Hybrid+H-SAM":   lambda: build_hybrid("hsam"),
    "EfficientNetV2": lambda: build_single("effnet"),
    "MobileNetV2":    lambda: build_single("mobilenet"),
    "ViT-B/16":       lambda: build_single("vit"),
}
CV_MODELS = ["MoViT-SAM", "Hybrid+Adam", "Hybrid+H-SAM", "EfficientNetV2"]
PROPOSED  = "MoViT-SAM"

def n_params(m):
    return (int(sum(np.prod(w.shape) for w in m.trainable_weights)),
            int(sum(np.prod(w.shape) for w in m.weights)))

def inner_split(idx, seed=SEED):
    """Carve a patient-disjoint early-stopping set out of the TRAINING indices only."""
    gss = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC, random_state=seed)
    a, b = next(gss.split(idx, labels[idx], groups[idx]))
    return idx[a], idx[b]

def fit(name, tr_idx, es_idx, seed=SEED, path_arr=None, label_arr=None, cw_idx=None):
    keras.backend.clear_session(); tf.random.set_seed(seed); np.random.seed(seed)
    model = MODELS[name]()
    cw = class_weights_for(cw_idx if cw_idx is not None else tr_idx) if path_arr is None else \
         {k: float(v) for k, v in zip(range(NUM_CLASSES),
          compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=label_arr[tr_idx]))}
    tr = make_ds(tr_idx, training=True, cw=cw, path_arr=path_arr, label_arr=label_arr)
    es = make_ds(es_idx, path_arr=path_arr, label_arr=label_arr)
    model.fit(tr, validation_data=es, epochs=EPOCHS, verbose=0, callbacks=[
        callbacks.EarlyStopping("val_accuracy", patience=PATIENCE, restore_best_weights=True),
        callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=3, min_lr=1e-6)])
    return model

def predict(model, idx, path_arr=None, label_arr=None):
    out = [model(x, training=False).numpy() for x, _, _ in make_ds(idx, path_arr=path_arr, label_arr=label_arr)]
    return np.vstack(out)

def macro_auc(y, p):
    try: return float(roc_auc_score(np.eye(NUM_CLASSES)[y], p, average="macro", multi_class="ovr"))
    except ValueError: return float("nan")

def metrics(y, p):
    pred = p.argmax(1)
    return {"accuracy": float((pred == y).mean()), "macro_auc": macro_auc(y, p),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0))}

In [ ]:
# ============================================================
# 11 - Five-fold stratified-group CV on all originals (identical folds for every model)
# ============================================================
sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(sgkf.split(paths, labels, groups))
for k, (tr, va) in enumerate(FOLDS):
    assert set(groups[tr]).isdisjoint(groups[va])

fold_rows, OOF = [], {}
for name in CV_MODELS:
    oof = np.zeros((len(df), NUM_CLASSES), np.float32)
    for k, (tr, va) in enumerate(FOLDS, 1):
        ck = CKPT_DIR / f"{re.sub(r'[^A-Za-z0-9]+', '_', name)}_fold{k}.npz"
        if ck.exists():
            p = np.load(ck)["prob"]
        else:
            t0 = time.time()
            tr_in, es_in = inner_split(tr, seed=SEED + k)
            model = fit(name, tr_in, es_in, seed=SEED + k, cw_idx=tr_in)
            p = predict(model, va)
            np.savez(ck, prob=p, idx=va)
            print(f"[{name}] fold {k}: {(time.time() - t0) / 60:.1f} min")
        oof[va] = p
        fold_rows.append({"model": name, "fold": k, **metrics(labels[va], p)})
    OOF[name] = oof
    np.save(RESULTS_DIR / f"oof_{re.sub(r'[^A-Za-z0-9]+', '_', name)}.npy", oof)

folds_df = pd.DataFrame(fold_rows)
folds_df.to_csv(RESULTS_DIR / "cv_per_fold_all_models.csv", index=False)
print(folds_df.to_string(index=False))

In [ ]:
# ============================================================
# 12 - CV summary: Table 3 (per fold), Table 4 (paired t-tests), pooled OOF
# ============================================================
def mean_sd_ci(x):
    x = np.asarray(x, float); m, sd = x.mean(), x.std(ddof=1)
    lo, hi = sp_stats.t.interval(0.95, len(x) - 1, loc=m, scale=sd / np.sqrt(len(x)))
    return m, sd, lo, hi

t3 = folds_df[folds_df.model == PROPOSED][["fold", "accuracy", "macro_auc", "macro_f1"]].copy()
summ = {c: mean_sd_ci(t3[c]) for c in ["accuracy", "macro_auc", "macro_f1"]}
t3 = t3.round(4).astype(object)
t3.loc[len(t3)] = ["Mean ± SD", *[f"{summ[c][0]:.4f} ± {summ[c][1]:.4f}" for c in summ]]
t3.loc[len(t3)] = ["95% CI (t)", *[f"[{summ[c][2]:.3f}, {summ[c][3]:.3f}]" for c in summ]]
print("TABLE 3 - MoViT-SAM five-fold patient-level CV\n", t3.to_string(index=False))
t3.to_csv(RESULTS_DIR / "table3_movit_sam_folds.csv", index=False)

base = folds_df[folds_df.model == PROPOSED].sort_values("fold")
others = [m for m in CV_MODELS if m != PROPOSED]
t4 = []
for name in CV_MODELS:
    d = folds_df[folds_df.model == name].sort_values("fold")
    row = {"Model": name, "Acc mean": d.accuracy.mean(), "Acc SD": d.accuracy.std(ddof=1),
           "Macro-AUC": d.macro_auc.mean(), "Macro-F1": d.macro_f1.mean()}
    if name != PROPOSED:
        p = sp_stats.ttest_rel(base.accuracy.values, d.accuracy.values).pvalue
        row["p (paired t)"] = p
        row["p (Bonferroni)"] = min(1.0, p * len(others))
    t4.append(row)
t4 = pd.DataFrame(t4).round(4)
print("TABLE 4 - CV comparison vs MoViT-SAM (paired t-test over folds; Bonferroni for",
      len(others), "comparisons)\n", t4.to_string(index=False))
t4.to_csv(RESULTS_DIR / "table4_cv_models.csv", index=False)

def bootstrap_ci(y, p, fn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        v = fn(y[i], p[i])
        if not np.isnan(v): vals.append(v)
    return np.percentile(vals, [2.5, 97.5])

P_oof = OOF[PROPOSED]
pooled = metrics(labels, P_oof)
acc_ci = bootstrap_ci(labels, P_oof, lambda y, p: float((p.argmax(1) == y).mean()))
auc_ci = bootstrap_ci(labels, P_oof, macro_auc)

pt = pd.DataFrame(P_oof).groupby(groups).mean()
pt_y = pd.Series(labels).groupby(groups).first().loc[pt.index].values
patient_level = metrics(pt_y, pt.values)

POOLED = {"pooled_oof": pooled, "acc_ci": acc_ci.tolist(), "macro_auc_ci": auc_ci.tolist(),
          "patient_level": patient_level, "n_images": len(df), "n_patients": int(len(pt))}
print(json.dumps(POOLED, indent=2))
(RESULTS_DIR / "pooled_oof_summary.json").write_text(json.dumps(POOLED, indent=2))

In [ ]:
# ============================================================
# 13 - Per-class performance (Table 5), confusion matrix (Fig 3), ROC/PR (Fig 5)
# ============================================================
pred_oof = P_oof.argmax(1)
rep = classification_report(labels, pred_oof, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
                            output_dict=True, zero_division=0)
Y1 = np.eye(NUM_CLASSES)[labels]
roc_c, pr_c = {}, {}
for i, c in enumerate(CLASS_NAMES):
    roc_c[c] = roc_auc_score(Y1[:, i], P_oof[:, i]) if Y1[:, i].any() else np.nan
    pr_c[c]  = average_precision_score(Y1[:, i], P_oof[:, i]) if Y1[:, i].any() else np.nan
t5 = pd.DataFrame([{"Class": c, "Precision": rep[c]["precision"], "Recall": rep[c]["recall"],
                    "F1": rep[c]["f1-score"], "Support": int(rep[c]["support"]),
                    "ROC-AUC": roc_c[c], "PR-AUC": pr_c[c]} for c in CLASS_NAMES])
t5.loc[len(t5)] = ["Macro avg", rep["macro avg"]["precision"], rep["macro avg"]["recall"],
                   rep["macro avg"]["f1-score"], len(df), np.nanmean(list(roc_c.values())),
                   np.nanmean(list(pr_c.values()))]
print("TABLE 5\n", t5.round(3).to_string(index=False))
t5.round(4).to_csv(RESULTS_DIR / "table5_per_class.csv", index=False)

cm = confusion_matrix(labels, pred_oof, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
sns.heatmap(cm / cm.sum(1, keepdims=True).clip(min=1) * 100, annot=True, fmt=".1f", cmap="Blues",
            ax=ax[1], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
for a, t in zip(ax, ["Counts", "Row %"]): a.set(title=t, xlabel="Predicted", ylabel="True")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig3_confusion.png", dpi=300); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for i, c in enumerate(CLASS_NAMES):
    if not Y1[:, i].any(): continue
    f, t, _ = roc_curve(Y1[:, i], P_oof[:, i]); ax[0].plot(f, t, label=f"{c} ({roc_c[c]:.3f})")
    pr, rc, _ = precision_recall_curve(Y1[:, i], P_oof[:, i]); ax[1].plot(rc, pr, label=f"{c} ({pr_c[c]:.3f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(xlabel="1 - Specificity", ylabel="Sensitivity", title="ROC (pooled OOF)")
ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall (pooled OOF)")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig5_roc_pr.png", dpi=300); plt.close()

In [ ]:
# ============================================================
# 14 - Calibration: cross-fitted temperature scaling, ECE, reliability diagram (Fig 4)
# ============================================================
# For each fold k, T is fitted by minimising NLL on the OOF predictions of the OTHER folds and
# applied to fold k, so no prediction is calibrated with a temperature fitted on itself.
def apply_T(p, T):
    z = np.log(np.clip(p, 1e-12, 1.0)) / T
    z -= z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_T(p, y):
    nll = lambda T: -np.mean(np.log(np.clip(apply_T(p, T)[np.arange(len(y)), y], 1e-12, 1)))
    return float(minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x)

def ece(p, y, n_bins=10):
    conf, pred = p.max(1), p.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0; pts = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            acc, cf = (pred[m] == y[m]).mean(), conf[m].mean()
            e += m.mean() * abs(acc - cf); pts.append((cf, acc))
    return float(e), np.array(pts)

P_cal = np.zeros_like(P_oof); Ts = []
for tr, va in FOLDS:
    T = fit_T(P_oof[tr], labels[tr]); Ts.append(T)
    P_cal[va] = apply_T(P_oof[va], T)
ece_before, pts_b = ece(P_oof, labels)
ece_after,  pts_a = ece(P_cal, labels)
CAL = {"T_per_fold": Ts, "T_mean": float(np.mean(Ts)), "ece_before": ece_before, "ece_after": ece_after,
       "argmax_unchanged": bool((P_cal.argmax(1) == P_oof.argmax(1)).all())}
print(json.dumps(CAL, indent=2))
(RESULTS_DIR / "calibration.json").write_text(json.dumps(CAL, indent=2))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(pts_b[:, 0], pts_b[:, 1], "o-", color="tab:orange", label=f"Before (ECE={ece_before:.3f})")
ax.plot(pts_a[:, 0], pts_a[:, 1], "s-", color="tab:green", label=f"After, cross-fitted T (ECE={ece_after:.3f})")
ax.set(xlabel="Mean predicted confidence", ylabel="Empirical accuracy", xlim=(0, 1), ylim=(0, 1))
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig4_reliability.png", dpi=300); plt.close()

In [ ]:
# ============================================================
# 15 - Clinical evaluation: binary Mpox screening, PPV/NPV (Table 7), DCA (Fig 6), error taxonomy
# ============================================================
yb = (labels == MPOX_IDX).astype(int)
sb = P_cal[:, MPOX_IDX]
b_auc = roc_auc_score(yb, sb)
b_auc_ci = bootstrap_ci(yb, sb, lambda y, s: roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan)
fpr, tpr, thr = roc_curve(yb, sb)

arg_pos = (P_cal.argmax(1) == MPOX_IDX)
sens_arg = (arg_pos & (yb == 1)).sum() / yb.sum()
spec_arg = (~arg_pos & (yb == 0)).sum() / (yb == 0).sum()
ops = [{"operating point": "argmax", "sensitivity": sens_arg, "specificity": spec_arg}]
for s in SENS_TARGETS:
    i = int(np.argmax(tpr >= s))
    ops.append({"operating point": f"sens>={s:.2f}", "threshold": thr[i],
                "sensitivity": tpr[i], "specificity": 1 - fpr[i]})
ops = pd.DataFrame(ops); print(ops.round(3).to_string(index=False))
ops.to_csv(RESULTS_DIR / "screening_operating_points.csv", index=False)

def ppv_npv(se, sp, prev):
    ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
    npv = sp * (1 - prev) / (sp * (1 - prev) + (1 - se) * prev)
    return ppv, npv

prev_list = PREVALENCES + [float(yb.mean())]
t7 = pd.DataFrame([{"Prevalence": f"{p*100:.1f}%", **dict(zip(["PPV", "NPV"], ppv_npv(sens_arg, spec_arg, p)))}
                   for p in prev_list]).round(3)
print("TABLE 7 (argmax operating point)\n", t7.to_string(index=False))
t7.to_csv(RESULTS_DIR / "table7_ppv_npv.csv", index=False)

def net_benefit(y, s, pt):
    pos = s >= pt; n = len(y)
    tp, fp = (pos & (y == 1)).sum(), (pos & (y == 0)).sum()
    return tp / n - fp / n * pt / (1 - pt)
prev = yb.mean()
dca = pd.DataFrame({"threshold": DCA_THRESHOLDS,
                    "model": [net_benefit(yb, sb, t) for t in DCA_THRESHOLDS],
                    "treat_all": [prev - (1 - prev) * t / (1 - t) for t in DCA_THRESHOLDS],
                    "treat_none": 0.0})
dca.to_csv(RESULTS_DIR / "dca.csv", index=False)
better = dca[(dca.model > dca.treat_all) & (dca.model > 0)].threshold
DCA_RANGE = (float(better.min()), float(better.max())) if len(better) else None
print("Net benefit above treat-all and treat-none for thresholds:", DCA_RANGE,
      "(check the range is contiguous in dca.csv)")
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(dca.threshold, dca.model, "b-", lw=2, label="MoViT-SAM")
ax.plot(dca.threshold, dca.treat_all, "k--", label="Screen all")
ax.axhline(0, color="k", lw=1, label="Screen none")
ax.set(xlabel="Threshold probability", ylabel="Net benefit", ylim=(-0.05, max(0.05, prev * 1.1)))
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig6_dca.png", dpi=300); plt.close()

err = pred_oof != labels
dangerous = int(((labels == MPOX_IDX) & (pred_oof == CLASS_NAMES.index("Healthy"))).sum())
moderate  = int((err & ((labels == MPOX_IDX) | (pred_oof == MPOX_IDX))).sum()) - dangerous
benign    = int((err & (labels != MPOX_IDX) & (pred_oof != MPOX_IDX)).sum())
CLIN = {"binary_auc": b_auc, "binary_auc_ci": b_auc_ci.tolist(), "sens_argmax": sens_arg,
        "spec_argmax": spec_arg, "dca_positive_range": DCA_RANGE,
        "errors": {"dangerous_mpox_to_healthy": dangerous, "moderate": moderate, "benign": benign,
                   "total": int(err.sum())}}
print(json.dumps(CLIN, indent=2, default=float))
(RESULTS_DIR / "clinical.json").write_text(json.dumps(CLIN, indent=2, default=float))

In [ ]:
# ============================================================
# 16 - Leakage quantification (Table 2)
# ============================================================
# Protocol A: image-level stratified 85/15 split that ignores patient identity.
# Protocol B: 6 augmented copies per original are materialised first (755 x 7 = 5,285 images),
#             then split 85/15 at image level: the leakage pattern of prior MSLD v2.0 studies.
# Both use the MoViT-SAM architecture and training recipe; early stopping uses an inner split
# drawn in the same (leaky) way from the training portion, as a naive pipeline would.
leak_rows = [{"Protocol": "Patient-level 5-fold CV (leakage-free)", "Contamination": "0 patients",
              "Accuracy": pooled["accuracy"], "Macro-AUC": pooled["macro_auc"]}]

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
a_tr, a_te = next(sss.split(paths, labels))
a_tr_in, a_es = next(StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_FRAC,
                                            random_state=SEED).split(paths[a_tr], labels[a_tr]))
mA = fit(PROPOSED, a_tr[a_tr_in], a_tr[a_es], cw_idx=a_tr[a_tr_in])
pA = predict(mA, a_te)
shared_pat = len(set(groups[a_tr]) & set(groups[a_te]))
mA_ = metrics(labels[a_te], pA)
leak_rows.append({"Protocol": "Naive image-level split (patient overlap)",
                  "Contamination": f"{shared_pat} patients",
                  "Accuracy": mA_["accuracy"], "Macro-AUC": mA_["macro_auc"]})

AUG_DIR = Path(os.environ.get("MOVIT_AUG_DIR", "./protocolB_images")); AUG_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
b_paths, b_labels, b_src = [], [], []
for i, p in enumerate(paths):
    img = np.array(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    b_paths.append(p); b_labels.append(labels[i]); b_src.append(i)
    for c in range(AUG_COPIES_B):
        ang, sc = rng.uniform(-15, 15), rng.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), ang, sc)
        out = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE), borderMode=cv2.BORDER_REFLECT)
        if rng.random() < 0.5: out = out[:, ::-1]
        q = AUG_DIR / f"{i:05d}_{c}.jpg"
        if not q.exists(): Image.fromarray(np.ascontiguousarray(out)).save(q, quality=95)
        b_paths.append(str(q)); b_labels.append(labels[i]); b_src.append(i)
b_paths, b_labels, b_src = np.array(b_paths), np.array(b_labels), np.array(b_src)
print("Protocol B pool:", len(b_paths), "images")

rs = np.random.default_rng(SEED).permutation(len(b_paths))
n_te = int(round(0.15 * len(b_paths)))
b_te, b_tr = rs[:n_te], rs[n_te:]
b_tr_in, b_es = b_tr[:int(len(b_tr) * (1 - INNER_VAL_FRAC))], b_tr[int(len(b_tr) * (1 - INNER_VAL_FRAC)):]
leaked = np.isin(b_src[b_te], b_src[b_tr])
mB = fit(PROPOSED, b_tr_in, b_es, path_arr=b_paths, label_arr=b_labels)
pB = predict(mB, b_te, path_arr=b_paths, label_arr=b_labels)
mB_ = metrics(b_labels[b_te], pB)
leak_rows.append({"Protocol": "Augment-before-split (prior-study style)",
                  "Contamination": f"{int(leaked.sum())}/{len(b_te)} test images ({leaked.mean()*100:.1f}%)",
                  "Accuracy": mB_["accuracy"], "Macro-AUC": mB_["macro_auc"]})

t2 = pd.DataFrame(leak_rows)
t2["Inflation vs clean (pp)"] = (t2.Accuracy - t2.Accuracy.iloc[0]) * 100
print("TABLE 2\n", t2.round(4).to_string(index=False))
t2.round(4).to_csv(RESULTS_DIR / "table2_leakage.csv", index=False)

In [ ]:
# ============================================================
# 17 - Single-run ablation on the held-out patient-level test set (Table 6)
# ============================================================
FINAL = {}
t6 = []
for name in ["MobileNetV2", "EfficientNetV2", "ViT-B/16", "Hybrid+Adam", "MoViT-SAM", "Hybrid+H-SAM"]:
    m = fit(name, train_idx, val_idx, cw_idx=train_idx)
    p = predict(m, test_idx)
    tr_n, tot_n = n_params(m)
    t6.append({"Variant": name, "Optimiser": {"MoViT-SAM": "SAM", "Hybrid+H-SAM": "H-SAM"}.get(name, "Adam"),
               "Test accuracy": metrics(labels[test_idx], p)["accuracy"],
               "Macro-AUC": metrics(labels[test_idx], p)["macro_auc"],
               "Trainable params": tr_n, "Total params": tot_n})
    if name == PROPOSED:
        FINAL["model"] = m; FINAL["test_prob"] = p
        m.inner.save_weights(str(MODELS_DIR / "movit_sam_final.weights.h5"))
t6 = pd.DataFrame(t6)
print(f"TABLE 6 (held-out test: {len(test_idx)} images, {len(G[2])} patients)\n", t6.round(4).to_string(index=False))
t6.round(4).to_csv(RESULTS_DIR / "table6_ablation.csv", index=False)

In [ ]:
# ============================================================
# 18 - TFLite export: FP32 and INT8 dynamic-range quantisation (Table 8)
# ============================================================
# INT8 dynamic range = Optimize.DEFAULT WITHOUT a representative dataset (weights int8,
# activations float). The TFLite model takes raw [0,255] float32 RGB, 224x224; output index 0
# is Monkeypox. These facts are written to model_card.json for the deployment and
# external-validation code.
inner = FINAL["model"].inner
FP32_PATH, INT8_PATH = MODELS_DIR / "movit_sam_fp32.tflite", MODELS_DIR / "movit_sam_int8_dynamic.tflite"

def convert(path, quantise):
    conv = tf.lite.TFLiteConverter.from_keras_model(inner)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
    if quantise: conv.optimizations = [tf.lite.Optimize.DEFAULT]
    path.write_bytes(conv.convert())
    return path.stat().st_size / 2**20

def bench(path, warmup=5):
    itp = tf.lite.Interpreter(model_path=str(path)); itp.allocate_tensors()
    di, do = itp.get_input_details()[0], itp.get_output_details()[0]
    lat, pr = [], []
    for i in test_idx:
        x = np.array(Image.open(paths[i]).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), np.float32)[None]
        itp.set_tensor(di["index"], x)
        t0 = time.perf_counter(); itp.invoke(); lat.append((time.perf_counter() - t0) * 1000)
        pr.append(itp.get_tensor(do["index"])[0])
    lat = np.array(lat[warmup:]) if len(lat) > warmup else np.array(lat)
    pr = np.array(pr)
    return {"accuracy": float((pr.argmax(1) == labels[test_idx]).mean()),
            "mean_ms": float(lat.mean()), "sd_ms": float(lat.std()), "p95_ms": float(np.percentile(lat, 95))}

s32, s8 = convert(FP32_PATH, False), convert(INT8_PATH, True)
b32, b8 = bench(FP32_PATH), bench(INT8_PATH)
t8 = pd.DataFrame([{"Format": "TFLite FP32", "Size (MB)": s32, **b32},
                   {"Format": "TFLite INT8 (dynamic range)", "Size (MB)": s8, **b8}]).round(3)
t8["Compression"] = (s32 / t8["Size (MB)"]).round(2)
print("TABLE 8 - benchmark hardware:", HARDWARE.get("cpu_model", HARDWARE["processor"]), "\n", t8.to_string(index=False))
t8.to_csv(RESULTS_DIR / "table8_tflite.csv", index=False)

(MODELS_DIR / "labels.txt").write_text("\n".join(CLASS_NAMES))
(MODELS_DIR / "model_card.json").write_text(json.dumps({
    "input": {"shape": [1, IMG_SIZE, IMG_SIZE, 3], "dtype": "float32", "range": "raw RGB [0,255]",
              "resize": "PIL bilinear to 224x224"},
    "labels": CLASS_NAMES, "mpox_index": MPOX_IDX,
    "calibration_temperature": CAL["T_mean"], "hardware_benchmark": HARDWARE}, indent=2))

In [ ]:
# ============================================================
# 19 - Context table of published results (Table 10)
# ============================================================
# Figures are as reported by each study; protocols differ, so this is context, not a benchmark.
t10 = pd.DataFrame([
    ["ResNet-18 (Nayak et al.)", 2023, "CNN", "MSLD-like", 2, "Random hold-out", "99.49", "N/R"],
    ["Swin Transformer (Vuran et al.)", 2025, "Transformer", "MSLD v2.0", 6, "Random hold-out", "93.71", "N/R"],
    ["ViT (Vuran et al.)", 2025, "Transformer", "MSLD v2.0", 6, "Random hold-out", "93.10", "N/R"],
    ["Transfer CNNs (Ali et al.)", 2024, "CNN ensemble", "MSLD v2.0", 6, "Random hold-out", "83.59", "N/R"],
    ["MPXV-CNN (Thieme et al.)", 2023, "CNN", "139,198 images", 2, "Clinical split", "N/R", "0.966"],
    ["MoViT-SAM (this study)", 2026, "CNN+Transformer+SAM", "MSLD v2.0", 6, "Patient-level 5-fold CV",
     f"{summ['accuracy'][0]*100:.2f} ± {summ['accuracy'][1]*100:.2f}", f"{summ['macro_auc'][0]:.3f}"],
], columns=["Model", "Year", "Architecture", "Dataset", "Classes", "Validation", "Accuracy (%)", "AUC"])
print(t10.to_string(index=False))
t10.to_csv(RESULTS_DIR / "table10_context.csv", index=False)

In [ ]:
# ============================================================
# 20 - Summary of every number used in the manuscript
# ============================================================
SUMMARY = {
    "hardware": HARDWARE,
    "dataset": {"images": len(df), "patients": int(df.patient.nunique()),
                "test_images": int(len(test_idx)), "test_patients": len(G[2])},
    "cv_movit_sam": {k: {"mean": v[0], "sd": v[1], "ci95": [v[2], v[3]]} for k, v in summ.items()},
    "pooled": POOLED, "calibration": CAL, "clinical": CLIN,
    "leakage": t2.to_dict("records"), "tflite": t8.to_dict("records"),
}
(RESULTS_DIR / "manuscript_numbers.json").write_text(json.dumps(SUMMARY, indent=2, default=float))
print("Wrote", RESULTS_DIR / "manuscript_numbers.json")
for f in sorted(RESULTS_DIR.iterdir()): print("  ", f.name)

In [ ]:
# ============================================================
# 21 - External validation on MSLD v1.0 (inference only) + Fig. 7
# ============================================================
# Input handling and label order come from model_card.json written in Section 18; nothing is
# tuned on this dataset. MSLD v1.0 images that are near-duplicates (pHash) of any MSLD v2.0
# original are flagged, and metrics are reported on all images and on the non-overlapping subset.
import glob, zipfile
from sklearn.metrics import accuracy_score
MODEL_DIR, RES_DIR, V2_ROOT = MODELS_DIR, RESULTS_DIR, DATA_ROOT
V1_DIR = Path(os.environ.get("MOVIT_V1_DIR", "./data/msld_v1"))
if not TINY_TEST and not glob.glob(str(V1_DIR / "**" / "Monkey Pox"), recursive=True) \
        and not glob.glob("/kaggle/input/**/Monkey Pox", recursive=True):
    V1_DIR.mkdir(parents=True, exist_ok=True)
    rc = os.system(f'kaggle datasets download -d nafin59/monkeypox-skin-lesion-dataset --unzip -p "{V1_DIR}"')
    if rc != 0:
        raise RuntimeError("MSLD v1.0 download failed: add the nafin59 'monkeypox-skin-lesion-dataset' as an input.")
cands = [Path(p) for p in glob.glob(str(V1_DIR / "**" / "Monkey Pox"), recursive=True)
         + glob.glob("/kaggle/input/**/Monkey Pox", recursive=True) if "ugment" not in p]
assert cands, "MSLD v1.0 not found."
cands = [c for c in cands if "Original" in str(c)] or cands
V1_ROOT = cands[0].parent
print("MSLD v1.0 originals:", V1_ROOT)

N_BOOT, SEED, THRESHOLD, PHASH_THRESHOLD = 2000, 42, 0.5, 8
card = json.loads((MODEL_DIR / "model_card.json").read_text())
LABELS, MPOX = card["labels"], card["mpox_index"]
assert LABELS[MPOX] == "Monkeypox" and card["input"]["range"].startswith("raw RGB [0,255]")
SIZE = card["input"]["shape"][1]
MODELS = {n: MODEL_DIR / f for n, f in [("FP32", "movit_sam_fp32.tflite"),
                                        ("INT8 dynamic range", "movit_sam_int8_dynamic.tflite")]
          if (MODEL_DIR / f).exists()}

def imgs(d): return sorted(p for p in glob.glob(str(d / "*")) if p.lower().endswith((".jpg", ".jpeg", ".png")))
mpox, others = imgs(V1_ROOT / "Monkey Pox"), imgs(V1_ROOT / "Others")
files = np.array(mpox + others); y = np.array([1] * len(mpox) + [0] * len(others))
print(f"MSLD v1.0: {len(mpox)} Monkeypox, {len(others)} Others")

v2 = [p for p in glob.glob(str(V2_ROOT / "**" / "*"), recursive=True)
      if p.lower().endswith((".jpg", ".jpeg", ".png")) and "augment" not in p.lower()]
v2h = [imagehash.phash(Image.open(p).convert("RGB")) for p in v2]
v1h = [imagehash.phash(Image.open(p).convert("RGB")) for p in files]
overlap = np.array([any(h - g <= PHASH_THRESHOLD for g in v2h) for h in v1h])
print(f"Scanned {len(v2)} MSLD v2.0 original files; v1.0 images with a near-duplicate in v2.0: "
      f"{overlap.sum()}/{len(files)} (Monkeypox {overlap[y == 1].sum()}, Others {overlap[y == 0].sum()})")

def load(p):  # identical to training: RGB, 224x224, float32 in [0,255]
    return np.array(Image.open(p).convert("RGB").resize((SIZE, SIZE)), np.float32)[None]

def run(path):
    itp = tf.lite.Interpreter(model_path=str(path)); itp.allocate_tensors()
    di, do = itp.get_input_details()[0], itp.get_output_details()[0]
    out = []
    for f in files:
        itp.set_tensor(di["index"], load(f)); itp.invoke(); out.append(itp.get_tensor(do["index"])[0])
    return np.array(out)

def summarise(yy, prob):
    s = prob[:, MPOX]; pred = (s >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
    rng = np.random.default_rng(SEED); b = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(yy), len(yy))
        if 0 < yy[i].sum() < len(i): b.append(roc_auc_score(yy[i], s[i]))
    return {"n": int(len(yy)), "n_mpox": int(yy.sum()), "auc": roc_auc_score(yy, s),
            "auc_ci95": np.percentile(b, [2.5, 97.5]).tolist(), "accuracy": accuracy_score(yy, pred),
            "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
            "ppv": tp / (tp + fp) if tp + fp else float("nan"), "npv": tn / (tn + fn) if tn + fn else float("nan"),
            "tp_fp_tn_fn": [int(tp), int(fp), int(tn), int(fn)]}

RESULTS = {"v1_root": str(V1_ROOT), "v2_files_scanned": len(v2),
           "overlap_with_msld_v2": int(overlap.sum()),
           "overlap_mpox": int(overlap[y == 1].sum()), "overlap_others": int(overlap[y == 0].sum()),
           "threshold": THRESHOLD}
PROBS = {}
for name, path in MODELS.items():
    PROBS[name] = run(path)
    keep = ~overlap
    RESULTS[name] = {"all_images": summarise(y, PROBS[name]),
                     "non_overlapping_only": summarise(y[keep], PROBS[name][keep])
                                             if keep.sum() and 0 < y[keep].sum() < keep.sum() else None}
print(json.dumps(RESULTS, indent=2, default=float))
(RES_DIR / "external_validation_results.json").write_text(json.dumps(RESULTS, indent=2, default=float))

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for name, prob in PROBS.items():
    r = RESULTS[name]["all_images"]; f, t, _ = roc_curve(y, prob[:, MPOX])
    ax.plot(f, t, lw=2, label=f"{name}, all (AUC {r['auc']:.3f} [{r['auc_ci95'][0]:.3f}-{r['auc_ci95'][1]:.3f}])")
    r2 = RESULTS[name]["non_overlapping_only"]
    if r2:
        f, t, _ = roc_curve(y[~overlap], prob[~overlap, MPOX])
        ax.plot(f, t, lw=1.5, ls=":", label=f"{name}, non-overlapping (AUC {r2['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="1 - Specificity", ylabel="Sensitivity", title=f"MoViT-SAM on MSLD v1.0 (n={len(y)})")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(RES_DIR / "fig7_external_roc.png", dpi=300); plt.close()

In [ ]:
# ============================================================
# 22 - Bundle every file the manuscript needs into for_paper.zip
# ============================================================
want = ["manuscript_numbers.json", "external_validation_results.json", "dca.csv",
        "fig3_confusion.png", "fig4_reliability.png", "fig5_roc_pr.png", "fig6_dca.png", "fig7_external_roc.png",
        "table2_leakage.csv", "table4_cv_models.csv", "table5_per_class.csv", "screening_operating_points.csv"]
bundle = Path(os.environ.get("MOVIT_BUNDLE", "for_paper.zip"))
with zipfile.ZipFile(bundle, "w") as z:
    for f in want:
        if (RES_DIR / f).exists(): z.write(RES_DIR / f, f)
        else: print("missing:", f)
print("Download this file from the Output panel:", bundle.resolve())